In [2]:
#!pip install lifelines

In [1]:
import numpy as np
import pandas as pd
import lifelines
from tqdm import tqdm
from lifelines.datasets import load_regression_dataset
from lifelines import CoxPHFitter
from scipy.optimize import minimize

In [2]:
# 表10.1 のデータセット
df = pd.DataFrame(
    {
        "対照群": [
            1,
            1,
            2,
            2,
            3,
            4,
            4,
            5,
            5,
            8,
            8,
            8,
            8,
            11,
            11,
            12,
            12,
            15,
            17,
            22,
            23,
        ],
        "実治療群": [
            6,
            6,
            6,
            6,
            7,
            9,
            10,
            10,
            11,
            13,
            16,
            17,
            19,
            20,
            22,
            23,
            25,
            32,
            32,
            34,
            35,
        ],
        "打ち切り": [1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0],
    }
)

In [3]:
# 表10.2の作成
tmp1 = (
    pd.cut(df["実治療群"], bins=[0, 6, 7, 10, 13, 16, 22, 23, np.inf], right=False)
    .value_counts()
    .sort_index()
)
tmp2 = (
    pd.cut(
        df["実治療群"] * (1 - df["打ち切り"]),
        bins=[0, 6, 7, 10, 13, 16, 22, 23, np.inf],
        right=False,
    )
    .value_counts()
    .sort_index()
)
deaths = (tmp1 - tmp2).to_frame().clip(lower=0).rename(columns={"count": "deaths"})
arrive = pd.DataFrame({"n": [21, 21, 17, 15, 12, 11, 7, 6]}, index=deaths.index)
table10_2 = pd.merge(arrive, deaths, left_index=True, right_index=True)
table10_2 = pd.merge(
    table10_2,
    ((table10_2["n"] - table10_2["deaths"]) / table10_2["n"])
    .cumprod()
    .to_frame()
    .rename(columns={0: "S(y)"}),
    left_index=True,
    right_index=True,
)

In [4]:
table10_2

,n,deaths,S(y)
実治療群,,,
"[0.0, 6.0)",21,0,1.000000
"[6.0, 7.0)",21,3,0.857143
"[7.0, 10.0)",17,1,0.806723
"[10.0, 13.0)",15,1,0.752941
"[13.0, 16.0)",12,1,0.690196
"[16.0, 22.0)",11,1,0.627451
"[22.0, 23.0)",7,1,0.537815
"[23.0, inf)",6,1,0.448179


In [5]:
# 表10.3の作成
survival_time = pd.concat([df["対照群"], df["実治療群"]])
survival_time = (
    survival_time.to_frame().reset_index(drop=True).rename(columns={0: "生存時間"})
)
survival_time["group"] = [0] * df.shape[0] + [1] * df.shape[0]

# 打ち切りデータの作成
# 1であれば非打ち切り、0であれば打ち切り
survival_time["打ち切り"] = [0] * len(df) + df["打ち切り"].tolist()
survival_time["打ち切り"] = 1 - survival_time["打ち切り"]

In [6]:
survival_time["group"]

0     0
1     0
2     0
3     0
4     0
5     0
6     0
7     0
8     0
9     0
10    0
11    0
12    0
13    0
14    0
15    0
16    0
17    0
18    0
19    0
20    0
21    1
22    1
23    1
24    1
25    1
26    1
27    1
28    1
29    1
30    1
31    1
32    1
33    1
34    1
35    1
36    1
37    1
38    1
39    1
40    1
41    1
Name: group, dtype: int64

In [18]:
# 係数の初期値
beta = np.array([0, 0])
threshold = 1e-5
# for iters in tqdm(range(100000)):
c = 0
while True:
    U0 = survival_time["打ち切り"] - survival_time["生存時間"] * np.exp(
        beta[0] + beta[1] * survival_time["group"]
    )
    U0 = U0.sum()

    U1 = (
        survival_time["打ち切り"]
        - survival_time["生存時間"]
        * np.exp(beta[0] + beta[1] * survival_time["group"])
        * survival_time["group"]
    )
    U1 = U1.sum()

    U = np.array([U0, U1])

    J11 = ((survival_time["打ち切り"] - 1).sum()) ** 2 + len(survival_time)
    J12 = (survival_time["打ち切り"] - 1).sum() * (
        ((survival_time["打ち切り"] - 1) * survival_time["group"]).sum()
    ) + survival_time["group"].sum()
    J22 = (((survival_time["打ち切り"] - 1) * survival_time["group"]).sum()) ** 2 + (
        survival_time["group"] ** 2
    ).sum()

    J = np.array([[J11, J12], [J12, J22]])
    beta_new = beta + np.linalg.inv(J) @ U
    c += 1
    if abs(beta_new[1] - beta[1]) < threshold:
        print(f"Finish: {c} 回目")
        break
    else:
        beta = beta_new.copy()

Finish: 99330 回目


In [19]:
beta

array([-13.67241411,  11.28559928])